# Notebook 02 — Pré-processamento do Dataset Clínico

**Objetivo:** Executar o pipeline de pré-processamento (normalização → segmentação → tokenização)
e gerar os arquivos CoNLL para anotação e treinamento NER.

**Saída:** `data/processed/prescricoes/` e `data/processed/pareceres/`

**Pipeline:**
1. `TextNormalizer` — Unicode NFC, espaços, datas ISO, PHI numérico
2. `ClinicalSegmenter` — segmentação por tipo (livre / template)
3. `WordTokenizer` — tokenização clínica com tratamento de abreviações
4. Exportação CoNLL + JSONL

---

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(".").resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing.pipeline import PreprocessingPipeline
from src.analysis.exploratory import load_csv

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
print(f"Saída: {PROCESSED_DIR}")

In [ ]:
# ── Pré-processar prescrições ─────────────────────────────────────────────────
PRESC_PATH = PROJECT_ROOT / "data" / "raw" / "prescricoes.csv"

if PRESC_PATH.exists():
    df_presc = load_csv(str(PRESC_PATH))
    print(f"{len(df_presc):,} prescrições carregadas")
    
    pipe_presc = PreprocessingPipeline(
        doc_type="prescricoes",
        mask_phi=True,
        expand_abbrev=False,    # manter abreviações para NER reconhecer
        min_tokens=3,
    )
    sentences_presc = pipe_presc.run(df_presc)
    
    out_dir = PROCESSED_DIR / "prescricoes"
    pipe_presc.to_conll(sentences_presc, out_dir / "prescricoes.conll")
    pipe_presc.to_jsonl(sentences_presc, out_dir / "prescricoes.jsonl")
else:
    print("⚠️  prescricoes.csv não encontrado")

In [ ]:
# ── Pré-processar pareceres ───────────────────────────────────────────────────
PARECERES_PATH = PROJECT_ROOT / "data" / "raw" / "pareceres.csv"

if PARECERES_PATH.exists():
    df_par = load_csv(str(PARECERES_PATH))
    print(f"{len(df_par):,} pareceres carregados")
    
    pipe_par = PreprocessingPipeline(
        doc_type="pareceres",
        mask_phi=True,
        min_tokens=3,
    )
    sentences_par = pipe_par.run(df_par)
    
    out_dir = PROCESSED_DIR / "pareceres"
    pipe_par.to_conll(sentences_par, out_dir / "pareceres.conll")
    pipe_par.to_jsonl(sentences_par, out_dir / "pareceres.jsonl")
else:
    print("⚠️  pareceres.csv não encontrado")

In [ ]:
# ── Inspeção de amostra processada ───────────────────────────────────────────
# Verificar qualidade da normalização e segmentação

if 'sentences_presc' in dir():
    print(f"Total de sentenças (prescrições): {len(sentences_presc):,}")
    print()
    
    for sent in sentences_presc[:3]:
        print("Texto:", sent.original_text[:100])
        print("Tokens:", sent.texts[:10], "...")
        print()

In [ ]:
# ── Divisão treino/dev/teste com Iterative Stratification ─────────────────────
# TODO: implementar após anotação manual
# 
# from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
#
# Referência: Schiezaro et al. (2026) — ganho de +0.0742 F1 sobre random split
# Ver: docs/07_decisoes/decisoes_preprocessamento.md

print("Divisão de dados: implementar após anotação. Ver src/preprocessing/ para detalhes.")